In [ ]:
# ============================================================
# Notebook 8: "Testing with Geometry"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg, stats

import statsmodels.api as sm

from regression_geometry.core import ColumnSpace, Projection
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

> *"The question is never whether the wall got closer — it's whether the closing was worth the cost."*

## The Question of Nested Models

Notebook 3 showed that R-squared always increases when you add predictors. Notebook 7 showed that this can lead to overfitting. So how do you decide whether the extra predictors ACTUALLY help — whether the improvement in R-squared is "real" or just noise?

The classical answer is the F-test. And the F-test has a beautiful geometric interpretation: it compares two projections.

In [ ]:
# Two nested models on Meridian
df = load_meridian()
y = df["salary"].values

# Restricted: salary ~ experience + education
X_restricted = sm.add_constant(df[["experience", "education"]].values)
model_restricted = sm.OLS(y, X_restricted).fit()

# Unrestricted: salary ~ experience + education + performance + gender
X_full = sm.add_constant(df[["experience", "education", "performance", "gender"]].values)
model_full = sm.OLS(y, X_full).fit()

print(f"Restricted  (experience + education):                  R-sq = {model_restricted.rsquared:.4f}")
print(f"Unrestricted (+ performance + gender):                 R-sq = {model_full.rsquared:.4f}")
print(f"Increase: {model_full.rsquared - model_restricted.rsquared:.4f}")
print("\nBut is that increase meaningful, or just noise?")

---
### 🛑 PREDICT FIRST

The unrestricted model adds performance and gender to the regression. R-squared went up. The column space expanded — the wall got bigger. The projection got closer to y.

**Before running the next cell, write your prediction below:**

If performance and gender had NO relationship with salary, would R-squared still increase? By how much? How do you distinguish a "real" improvement from a "noise" improvement?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Monte Carlo: R-squared increase under the null (extra predictors are noise)
rng = np.random.default_rng(42)
n_sims = 200
r2_increases = np.zeros(n_sims)

for i in range(n_sims):
    perf_perm = rng.permutation(df["performance"].values)
    gender_perm = rng.permutation(df["gender"].values)
    X_null = sm.add_constant(np.column_stack([
        df["experience"].values, df["education"].values,
        perf_perm, gender_perm]))
    r2_increases[i] = sm.OLS(y, X_null).fit().rsquared - model_restricted.rsquared

actual_increase = model_full.rsquared - model_restricted.rsquared

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(r2_increases, bins=30, color=SECONDARY, alpha=0.6, edgecolor="white")
ax.axvline(actual_increase, color=RESIDUAL, lw=2, ls="--",
           label=f"Actual = {actual_increase:.4f}")
ax.axvline(np.mean(r2_increases), color=COLUMN_SPACE, lw=1.5,
           label=f"Null mean = {np.mean(r2_increases):.4f}")
ax.set(xlabel="R-squared increase", ylabel="Count",
       title="R-squared increase: real data vs. noise")
ax.legend(fontsize=TICK_SIZE)
plt.tight_layout()
plt.show()

Even random predictors expand the column space and catch a little more of y by chance. The F-test asks: is the improvement you see bigger than what noise alone would produce?

## The F-Test as a Distance Between Projections

In [ ]:
# Two nested projections in 3D (n=3 for visibility)
x3_1 = np.array([2.0, 5.0, 8.0])
x3_2 = np.array([1.0, 3.0, 2.0])
y3 = np.array([6.5, 12.8, 19.2])

cs_restricted_3d = ColumnSpace(x3_1, add_intercept=True)
X3_full = np.column_stack([np.ones(3), x3_1, x3_2])
cs_full_3d = ColumnSpace(X3_full, add_intercept=False)

fig = viz.plot_nested_projections(cs_restricted_3d, cs_full_3d, y3,
    title="F-test: Comparing Two Projections")

The restricted model projects y onto a line. The unrestricted model projects y onto a plane that contains that line. The purple gap between the two projections is what the extra predictors explain.

The F-statistic is:

F = (||y_hat_full - y_hat_restricted||^2 / q) / (||e_full||^2 / (n - p))

- **Numerator:** squared distance between the two projections, normalized by q (extra predictors). How much closer the full projection got.
- **Denominator:** squared residual of the full model, normalized by (n - p). The noise level.

F is "improvement per extra predictor" divided by "noise per remaining degree of freedom." If the extra predictors are just noise, F is approximately 1. If real, F >> 1.

In [ ]:
# Compute the F-statistic manually from geometric quantities
cs_r = ColumnSpace(X_restricted, add_intercept=False)
cs_f = ColumnSpace(X_full, add_intercept=False)
proj_r = cs_r.project(y)
proj_f = cs_f.project(y)

sse_restricted = proj_r.sse
sse_full = proj_f.sse
q = cs_f.p - cs_r.p
df_resid = cs_f.n - cs_f.p

F_stat = ((sse_restricted - sse_full) / q) / (sse_full / df_resid)
p_value = 1 - stats.f.cdf(F_stat, q, df_resid)

print(f"SSE restricted: {sse_restricted:,.0f}")
print(f"SSE full:       {sse_full:,.0f}")
print(f"Extra predictors (q): {q}")
print(f"Residual df:    {df_resid}")
print(f"\nF-statistic:    {F_stat:.4f}")
print(f"p-value:        {p_value:.6f}")

In [ ]:
# Verify against statsmodels
f_test_result = model_full.f_test("x3 = 0, x4 = 0")

print(f"Our F-statistic:         {F_stat:.4f}")
print(f"statsmodels F-statistic: {float(f_test_result.fvalue):.4f}")
print(f"\nOur p-value:             {p_value:.6f}")
print(f"statsmodels p-value:     {float(f_test_result.pvalue):.6f}")
print(f"\nThey match. The F-test is a distance ratio between two projections.")

## "Theorem Without Algebra": The F-Test Diagram

The definitive one-picture proof: the restricted subspace (inner) is contained in the unrestricted subspace (outer). y is projected onto both. The **gap** between the two projections = numerator of F. The **residual** from the full model = denominator of F.

Under the null hypothesis (the extra predictors don't matter), y's component in the gap direction is pure noise. The F-statistic compares the size of that noise to the residual noise. If they're comparable, the extra predictors didn't help. If the gap is much larger than expected from noise, the extra predictors captured real signal.

---
### 🛑 PREDICT FIRST

You're about to see the F-test results on Meridian (restricted = experience + education, unrestricted = + performance + gender).

**Before running the next cell, write your prediction below:**

Will the F-test be significant? Which of the two extra variables (performance, gender) contributes more to the gap between projections?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# F-test on Meridian + individual t-tests
print(f"F-test (performance + gender jointly):")
print(f"  F = {F_stat:.2f}, p = {p_value:.6f}")
print(f"  {'Significant' if p_value < 0.05 else 'Not significant'} at alpha = 0.05")
print()
print(f"Individual t-tests:")
for i, name in enumerate(["const", "experience", "education", "performance", "gender"]):
    if name in ["performance", "gender"]:
        print(f"  {name:>12}: t = {model_full.tvalues[i]:>7.3f}, p = {model_full.pvalues[i]:.6f}")

The gap between projections tells us which variables carry signal after controlling for the others. The geometry made this visible before the p-value did.

<details>
<summary><b>Going Deeper:</b> The t-test as a special case of the F-test</summary>

When q = 1 (adding one predictor), F = t-squared. The F-test on one restriction is equivalent to squaring the t-statistic. Geometrically: the gap between projections has one dimension, so it's just a distance along a single direction. The t-statistic measures that distance normalized by the residual noise.

</details>

In [ ]:
# Verify F = t^2 when q = 1
t_gender = model_full.tvalues[4]
f_gender = model_full.f_test("x4 = 0")

print(f"t-statistic for gender:  {t_gender:.4f}")
print(f"t-squared:               {t_gender**2:.4f}")
print(f"F-statistic (q=1):       {float(f_gender.fvalue):.4f}")
print(f"\nF = t-squared. Same geometry, one dimension.")

## From Point Estimates to Regions

The projection gives you a single point — beta_hat. But beta_hat is a random variable. If you drew a different sample, you'd get a different beta_hat. The confidence ellipsoid is the region where beta_hat would fall under repeated sampling — the "cloud of shadows" from different datasets.

In [ ]:
# 95% confidence ellipse for experience and education coefficients
fig = viz.plot_confidence_ellipse(
    model_restricted.params,
    model_restricted.cov_params(),
    confidence=0.95,
    indices=(1, 2),
    title="Confidence Ellipse: experience vs. education"
)

The ellipse is elongated along the direction of highest uncertainty. If experience and education are correlated, the ellipse tilts — uncertainty in one coefficient is coupled to uncertainty in the other.

---
### 🛑 DIAGNOSE FIRST

Here is the covariance matrix of the coefficients from the Meridian regression (experience + education).

**Before seeing the visualization, answer:**

Will the confidence ellipse be elongated or roughly circular? In which direction will the long axis point? (Hint: look at the off-diagonal elements of the covariance matrix.)

---

In [ ]:
# Show the covariance matrix for diagnosis
cov_mat = model_restricted.cov_params()
print("Covariance matrix (rows/cols: const, experience, education):")
print(np.array2string(cov_mat, precision=4, suppress_small=True))
print(f"\nCorrelation between experience and education coefficients:")
print(f"  {cov_mat[1,2] / np.sqrt(cov_mat[1,1] * cov_mat[2,2]):.4f}")

In [ ]:
# Monte Carlo: watching the cloud of beta-hats form
cs_mc = ColumnSpace(X_restricted, add_intercept=False)
beta_true_mc = model_restricted.params
sigma_mc = np.sqrt(model_restricted.mse_resid)

if INTERACTIVE:
    display(viz.plot_monte_carlo_projections(cs_mc, beta_true_mc, sigma_mc,
        n_samples=20, title="Sampling Distribution of Projections"))
else:
    rng_mc = np.random.default_rng(42)
    betas_mc = np.zeros((200, len(beta_true_mc)))
    for i in range(200):
        eps = rng_mc.normal(0, sigma_mc, cs_mc.n)
        y_sample = X_restricted @ beta_true_mc + eps
        betas_mc[i] = cs_mc.project(y_sample).coefficients
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(betas_mc[:, 1], betas_mc[:, 2], s=10, color=PROJECTION,
               alpha=0.4, label="Sampled beta-hats")
    ax.plot(beta_true_mc[1], beta_true_mc[2], "*", color=RESPONSE_Y,
            markersize=15, zorder=5, label="True beta")
    ax.set(xlabel="beta_experience", ylabel="beta_education",
           title="Cloud of 200 beta-hats")
    ax.legend(fontsize=TICK_SIZE)
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

Each green point is a beta-hat from one sample. The cloud is the sampling distribution. The ellipse contains 95% of the cloud. This is what "confidence" means geometrically — the region where repeated projections land.

The F-test and the confidence ellipsoid are two views of the same geometry. The F-test asks whether the full model's beta-hat could plausibly have come from a world where the restricted model is true — whether the extra coefficients fall inside the "plausible region" around zero. Rejecting the null = the origin (for the extra coefficients) is outside the confidence ellipsoid.

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| F-test (comparing two projections) | `model.f_test(restriction_matrix)` |
| Confidence intervals | `model.conf_int()` |
| Covariance matrix of coefficients | `model.cov_params()` |

---

In [ ]:
# Demonstrate statsmodels F-test and confidence intervals
f_result = model_full.f_test("x3 = 0, x4 = 0")
print(f"F-test 'x3 = 0, x4 = 0': F = {float(f_result.fvalue):.4f}, p = {float(f_result.pvalue):.6f}")
print()
print("Confidence intervals (95%):")
ci = model_full.conf_int(alpha=0.05)
names = ["const", "experience", "education", "performance", "gender"]
for i, name in enumerate(names):
    print(f"  {name:>12}: [{ci[i, 0]:>10.3f}, {ci[i, 1]:>10.3f}]")

---
### ✍️ The Memo

You're writing a memo to Meridian's VP of HR. You've run a regression with experience, education, performance, and gender predicting salary. The VP asks: "Do performance ratings actually predict salary, or is that just noise?"

Write your answer in 3 sentences.

**Rules:** Do not use the words "F-test," "p-value," "null hypothesis," "confidence ellipsoid," or "projection." Translate the statistical result into a business claim.

---

In [ ]:
# YOUR MEMO:
#
#
#

In [ ]:
# Geometric Scoreboard — all 5 gauges active
cs_full_mer = ColumnSpace(X_full, add_intercept=False)
proj_full_mer = cs_full_mer.project(y)

scoreboard = GeometricScoreboard(
    proj=proj_full_mer,
    cs=cs_full_mer,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard.display()

All five gauges active. Compare to the restricted model: R-squared is slightly higher, theta slightly smaller, but kappa and leverage are barely affected. The extra predictors shifted the projection marginally closer to y — now you know HOW to test whether that shift was real.

---
### Summary

**What you learned:**
- The F-test compares two projections — one on a restricted column space, one on a larger space. The gap between them, normalized by residual noise, gives the F-statistic.
- A large gap means the extra predictors captured real signal, not just noise.
- Confidence ellipsoids show the region where beta-hat would fall under repeated sampling — the geometric face of statistical uncertainty.

**Key geometric insight:** ***The F-test measures whether the distance between two projections is surprising given the noise level.***

### Next

Notebook 9 puts all your geometric tools together. You'll face an unfamiliar dataset and diagnose it using only the geometric lens — the Scoreboard, the eigenvalue structure, the leverage plot, the projections. Can you read the instruments?

---